# Plots

In [ ]:
library(readr)
library(purrr)
library(dplyr)
library(stringr)
library(fuzzyjoin)
library(ggplot2)
library(tidyverse)
library(Matrix)
library(reshape2)
library(RColorBrewer)
library(rstatix)
library(emmeans)
library(tibble)
library(tidyr)

In [ ]:
options(repr.matrix.max.cols = Inf,  # show all columns
        repr.matrix.max.rows = 200)  # adjust rows as you like

## 0. Plot parameters

In [ ]:
out_dir = "/ceph.groups/mshahbazi.grp/rsakata/Figures/GFP_marker/output"
filter_dapi = TRUE

analysis_summary_files = c(
"/ceph.groups/mshahbazi.grp/rsakata/EXP56/output/setA_T3/4_plots/EXP56_setA_T3_analysis_summary.csv",
"/ceph.groups/mshahbazi.grp/rsakata/EXP58/output/setA_T2/4_plots/EXP58_setA_T2_analysis_summary.csv",
"/ceph.groups/mshahbazi.grp/rsakata/EXP58/output/setA_T3/4_plots/EXP58_setA_T3_analysis_summary.csv",
"/ceph.groups/mshahbazi.grp/rsakata/EXP62/output/setA_T2/4_plots/EXP62_setA_T2_analysis_summary.csv",
"/ceph.groups/mshahbazi.grp/rsakata/EXP62/output/setA_T3/4_plots/EXP62_setA_T3_analysis_summary.csv",
"/ceph.groups/mshahbazi.grp/rsakata/EXP63/output/setA_T3/4_plots/EXP63_setA_T3_analysis_summary.csv"
)

In [ ]:
# define const for visualization
FONT.SIZE <- 7
LABEL.FONT.SIZE <- 7
w <- 2 
h <- 2.5
LINE.W <- 0.5/2.141959

# Set geom defaults globally
update_geom_defaults("line",      list(linewidth = LINE.W))
update_geom_defaults("errorbar",  list(linewidth = LINE.W))
#update_geom_defaults("point",     list(size = LINE.W, stroke = LINE.W))

settheme <- theme_minimal() + 
  theme(
    text = element_text(family = "sans"), 
    panel.background = element_blank(),
    panel.grid.major = element_blank(), 
    panel.grid.minor = element_blank(),
    plot.background = element_blank(),
    axis.ticks = element_line(colour = "black", linewidth = LINE.W),
    axis.ticks.length = unit(0.1, "cm"), 
    axis.line = element_line(linewidth = LINE.W, colour = "black"),
    axis.title = element_text(size = FONT.SIZE),
    axis.text = element_text(colour = "black", size = FONT.SIZE),
    strip.text = element_text(size = FONT.SIZE), 
    strip.text.y.left = element_text(angle = 0, hjust = 1, size = FONT.SIZE),
    legend.position = "right",
    legend.title = element_text(size = FONT.SIZE), 
    legend.text = element_text(size = FONT.SIZE),
    legend.key.size = unit(0.3, "cm"),
    axis.text.x = element_text(colour = "black", angle = 0, size = LABEL.FONT.SIZE),
    title = element_text(size = FONT.SIZE) 
  )

In [ ]:
col_aneu = c("euploid"= "#D4D1B3","monosomy"="#109E9D","trisomy"="#F26B3B","complex"= "#886DB0")

col_condition_2 = c("control" = "#285F63", 
               "reversine" = "#CA4F33", 
               "mosaic"= "#E2A557")

col_condition = c("G_R"= "#5E5E5E","Grev_R"="#86AB30","Rrev_G"="#EB5951", "Grev_Rrev"="#F0A329")

col_GFP = c("TRUE"= "#86AB30","FALSE"="#8d8d8dff")

col_condition_3 = c("Developed" = "#285F62", 
               "Poor Quality" = "#CA4F33")



col_unspecified = "#5E5E5E"
col_GATA3 = "#489C9C"
col_NANOG = "#EA9542"
col_neg    = "#8d8d8dff"



## 1. Extract summary files

In [ ]:
# Read and combine all files into one dataframe
merged_df <- analysis_summary_files %>%
  map_dfr(read_csv)

In [ ]:
tbl <- merged_df %>%
  group_by( sample_name) %>%
  summarise(n_images = n_distinct(image), .groups = "drop") 
tbl

In [ ]:
tbl <- merged_df %>%
  group_by(exp, exp_sub) %>%
  summarise(n = n_distinct(image), .groups = "drop") 
tbl

## 2. Preprocess

In [ ]:
colnames(merged_df)

# Filter cells with higher dapi levels
if (filter_dapi) {
  merged_df <- merged_df %>% 
    filter(Mean_dapi < DAPI_thresh)
}

In [ ]:
unique(merged_df$sample_name)

In [ ]:

order_sample <- c(
  "D4_GR",  "D4_GrevRrev", "D4_GrevR", "D4_RrevG",
  "D6_GR_D", "D6_GR_F", "D6_GrevRrev_D", "D6_GrevRrev_F", 
  "D6_GrevR_D", "D6_GrevR_F", "D6_RrevG_D", "D6_RrevG_F")
merged_df <- merged_df %>%
  mutate(sample_name = factor(sample_name, levels = order_sample))

## Plot 

### C) Intensity threshold jitter

In [ ]:
library(ggplot2)
library(rlang)

plot_jitter <- function(
  data,
  x = sample_name,
  y = GATA3_norm,
  color = condition,
  out_dir,
  title = "",
  w = 5, h = 3,
  palette = NULL,
  hline_at = NULL,                 # <— add a dotted horizontal line at this y
  hline_color = "gray30",
  hline_size = 0.5,
  hline_lty =  "dashed"
) {
  x <- rlang::enquo(x); y <- rlang::enquo(y); color <- rlang::enquo(color)

  p <- ggplot(data, aes(x = fct_rev(!!x), y = !!y, color = !!color)) +
    geom_jitter(width = 0.2, size = 0.3, alpha = 0.6, na.rm = TRUE) +
    labs(x = "", y = "normalised intensity", title = title) +
    settheme +
    scale_y_continuous(limits = c(0, NA), expand = c(0, 0)) +
    coord_flip()+
    theme(legend.position = "none")

  if (!is.null(palette)) p <- p + scale_color_manual(values = palette)
  if (!is.null(hline_at)) p <- p + geom_hline(yintercept = hline_at, linetype = hline_lty,
                                              linewidth = hline_size, color = hline_color)

  ggsave(file.path(out_dir, sprintf("%s.pdf", title)), plot = p, width = w, height = h)
  p
}


### D) Intensity threshold_hist

In [ ]:
plot_hist <- function(
  data,
  x = GATA3_norm,
  facet = sample_name,
  out_dir,
  title = "GATA3_norm_hist",
  w = 5, h = 5,
  bins = 30,
  binwidth = NULL,
  fill = "#6CD1D4",
  outline = "gray10",
  linewidth = 0.1,
  free_y = TRUE,
  vline_at = NULL,              # <— vertical line at this x
  vline_color = "gray30",
  vline_size = 0.5,
  vline_lty = "dashed"
) {
  x     <- rlang::enquo(x)
  facet <- rlang::enquo(facet)

  p <- ggplot(data, aes(x = !!x)) +
    (if (!is.null(binwidth))
       geom_histogram(binwidth = binwidth, boundary = 0, closed = "left",
                                na.rm = TRUE, fill = fill, color = outline, linewidth = linewidth)
     else
       geom_histogram(bins = bins, na.rm = TRUE,
                                fill = fill, color = outline, linewidth = linewidth)) +
    labs(x = "normalised intensity", y = "Count", title = title) +
    settheme +
    scale_y_continuous(limits = c(0, NA), expand = c(0, 0),
      breaks = scales::pretty_breaks(n = 2)   # <— fewer ticks
    )  +
    facet_grid(rows = vars(!!facet),
                        scales = if (free_y) "free_y" else "fixed",
      switch = "y"            ) +
    theme(
      strip.text.y.left = element_text(angle = 0, hjust = 0),
      strip.background = element_blank(),
      strip.placement  = "outside",
     # plot.margin = margin(5.5, 5.5, 5.5, 35, "pt"),
      legend.position = "none"
    )

  if (!is.null(vline_at)) {
    p <- p + geom_vline(xintercept = vline_at,
                                 linetype = vline_lty,
                                 linewidth = vline_size,
                                 color = vline_color)
  }

  ggsave(file.path(out_dir, sprintf("%s.pdf", title)),
                  plot = p, width = w, height = h)
  p
}




In [ ]:
#dapi
w <- 4
h <- 3
options(repr.plot.width=w, repr.plot.height=h)

DAPI_thresh = 75

gghist <- plot_hist(
  data    = merged_df,
  x       = Mean_dapi,
  facet   = sample_name,
  out_dir = out_dir,
  binwidth = 0.1,
  title   = "D_Mean_dapi_hist",
  fill = "#362fffff",
  w = w, h = h,
  vline_at = DAPI_thresh
)
gghist

In [ ]:
#NANOG

NANOG_thresh = 0.2

w <- 3
h <- 2
options(repr.plot.width=w, repr.plot.height=h)
ggjitter <- plot_jitter(
  data = merged_df,
  x = sample_name,
  y = NANOG_norm,
  color = condition,
  out_dir = out_dir,
  title = "C_NANOG_norm_intensity",
  w = w, h = h,
  palette = col_condition,
  hline_at = NANOG_thresh
)
ggjitter

w <- 3
h <- 3
options(repr.plot.width=w, repr.plot.height=h)

gghist <- plot_hist(
  data    = merged_df,
  x       = NANOG_norm,
  facet   = sample_name,
  out_dir = out_dir,
  binwidth = 0.1,
  title   = "D_NANOG_norm_hist",
  fill = "#FF2F92",
  w = w, h = h,
  vline_at = NANOG_thresh
)
gghist


In [ ]:
w <- 3
h <- 2
options(repr.plot.width=w, repr.plot.height=h)

GATA3_thresh = 0.5

gg_jitter <- plot_jitter(
  data = merged_df,
  x = sample_name,
  y = GATA3_norm,
  color = condition,
  out_dir = out_dir,
  title = "C_GATA3_norm_intensity",
  w = w, h = h,
  palette = col_condition,
  hline_at = GATA3_thresh
)
gg_jitter

w <- 3
h <- 3
options(repr.plot.width=w, repr.plot.height=h)

gg_hist <- plot_hist(
  data    = merged_df,
  x       = GATA3_norm,
  facet   = sample_name,
  out_dir = out_dir,
  binwidth = 0.1,
  title   = "D_GATA3_norm_hist",
  fill = "#6CD1D4",
  w = w, h = h,
  vline_at = GATA3_thresh)
gg_hist

In [ ]:
w <- 3
h <- 2
options(repr.plot.width=w, repr.plot.height=h)

GFP_thresh = 1

gg_jitter <- plot_jitter(
  data = merged_df,
  x = sample_name,
  y = GFP_norm,
  color = condition,
  out_dir = out_dir,
  title = "C_GFP_norm_intensity",
  w = w, h = h,
  palette = col_condition,
  hline_at = GFP_thresh
)
gg_jitter

w <- 4
h <- 3
options(repr.plot.width=w, repr.plot.height=h)

gg_hist <- plot_hist(
  data    = merged_df,
  x       = GFP_norm,
  facet   = sample_name,
  out_dir = out_dir,
  binwidth = 0.1,
  title   = "D_GFP_norm_hist",
  fill = "#6EA537",
  w = w, h = h,
  vline_at = GFP_thresh
)
gg_hist

In [ ]:
title = "GATA3_NANOG_scatter"

w <- 5
h <- 4
options(repr.plot.width=w, repr.plot.height=h)

ggscatter = ggplot(merged_df, aes(x = NANOG_norm, y = GATA3_norm, color = condition)) +
  geom_point(alpha = 0.6, size = 0.6) +
  labs(x = " NANOG_norm", y = " GATA3_norm", title = title) +
  facet_wrap(~sample_name)+
  settheme+
   scale_color_manual(values=col_condition)

ggsave(plot = ggscatter, filename = sprintf("%s/%s.pdf",out_dir, title), w = w, h = h)
ggscatter

### E) %pos marker

In [ ]:
plot_pct_bar_points <- function(
  data,                               # e.g., summary_df
  pct = pct_GATA3,                    # <-- column with % values to plot
  sample = sample_name,               # sample/category column
  condition = condition,              # grouping/fill column
  out_dir,                    # folder to save (optional)
  title = NULL,                       # default built from pct col name if NULL
  palette = NULL,                     # named vector for fill
  w = 3, h = 2,
  y_max = 110,
  y_ticks = 5,
  bar_width = 0.6,
  point_size = 0.3,
  point_alpha = 0.7,
  jitter_width = 0.05
) {
  pct      <- enquo(pct)
  sample   <- enquo(sample)
  condition<- enquo(condition)

  # default title from pct column name if not supplied
  if (is.null(title)) {
    title <- paste0(as_label(pct), "+")
  }

  # per-sample means (by condition) for the chosen pct column
  means_df <- data %>%
    group_by(!!sample, !!condition) %>%
    summarise(mean_pct = mean(!!pct, na.rm = TRUE), .groups = "drop")

  # build plot (reverse sample order, flip coords)
  p <- ggplot(data, aes(x = fct_rev(!!sample), y = !!pct)) +
    geom_col(
      data = means_df,
      aes(y = mean_pct, fill = !!condition),
      width = bar_width
    ) +
    geom_point(
      size = point_size, alpha = point_alpha,
      position = position_jitter(width = jitter_width),
      na.rm = TRUE
    ) +
    labs(x = "", y = "% positive", title = title, fill = rlang::as_name(condition)) +
    settheme +
    scale_y_continuous(limits = c(0, y_max), expand = c(0, 0),
                       breaks = scales::pretty_breaks(y_ticks)) +
    coord_flip()+
    theme(legend.position = "none")

  if (!is.null(palette)) {
    p <- p + scale_fill_manual(values = palette)
  }

  safe_title <- gsub("[^[:alnum:]_\\-]+","_", title)
  ggsave(file.path(out_dir, sprintf("%s.pdf", safe_title)),
                  plot = p, width = w, height = h)
  options(repr.plot.width=w, repr.plot.height=h)
  p
}

In [ ]:
# if using new threshold 
# thresholds (edit if you want different cutoffs)
thr <- list(
  GATA3_norm = GATA3_thresh,
  NANOG_norm = NANOG_thresh,
  GFP_norm = GFP_thresh
)

summary_df <- merged_df %>%
  group_by(exp, image, sample_name, condition) %>%
  summarise(
    n = n(),
    pct_GATA3 = 100 * mean(GATA3_norm > thr$GATA3_norm, na.rm = TRUE),
    pct_NANOG = 100 * mean(NANOG_norm > thr$NANOG_norm, na.rm = TRUE),
    pct_GFP = 100 * mean(GFP_norm > thr$GFP_norm, na.rm = TRUE),
    pct_negative = 100 * mean(!(GFP_norm > thr$GFP_norm | NANOG_norm > thr$NANOG_norm | GATA3_norm > thr$GATA3_norm)),
    pct_double_GATA3_NANOG = 100 * mean(GATA3_norm > thr$GATA3_norm & NANOG_norm > thr$NANOG_norm),
    .groups = "drop"
  )
  
# Plot % GATA3+
plot_pct_bar_points(summary_df, pct = pct_GATA3, palette = col_condition, out_dir = out_dir, 
                    title = "E_pctGATA3+")

# Plot % NANOG+
plot_pct_bar_points(summary_df, pct = pct_NANOG, palette = col_condition,out_dir = out_dir,
                    title = "E_pctNANOG+")
# Plot % GFP+
plot_pct_bar_points(summary_df, pct = pct_GFP, palette = col_condition,out_dir = out_dir,
                    title = "E_pctGFP+")

# Plot % neg
plot_pct_bar_points(summary_df, pct = pct_negative, palette = col_condition,out_dir = out_dir, 
                    title = "E_pctnegative", y_max = 40)

# Plot % double+
plot_pct_bar_points(summary_df, pct = pct_double_GATA3_NANOG, palette = col_condition,out_dir = out_dir, 
                    title = "E_pct_GATA3+_NANOG+", y_max = 40)


In [ ]:
summary_df <- merged_df %>%
  group_by(exp, image, sample_name, condition) %>%
  summarise(
    n = n(),
    
    # Calculate simple percentages based on your existing boolean columns
    pct_GATA3 = 100 * mean(GATA3pos, na.rm = TRUE),
    pct_NANOG = 100 * mean(NANOGpos, na.rm = TRUE),
    pct_GFP   = 100 * mean(GFPpos,   na.rm = TRUE),
    
    # Calculate negative cells (None of the markers are positive)
    pct_negative = 100 * mean(!(GFPpos | NANOGpos | GATA3pos), na.rm = TRUE),
    
    # Calculate double positives (Both markers are positive)
    pct_double_GATA3_NANOG = 100 * mean(GATA3pos & NANOGpos, na.rm = TRUE),
    
    .groups = "drop"
  )

# Plot % GATA3+
plot_pct_bar_points(summary_df, pct = pct_GATA3, palette = col_condition, out_dir = out_dir, 
                    title = "E_pctGATA3+")
# Plot % NANOG+
plot_pct_bar_points(summary_df, pct = pct_NANOG, palette = col_condition,out_dir = out_dir,
                    title = "E_pctNANOG+")
# Plot % GFP+
plot_pct_bar_points(summary_df, pct = pct_GFP, palette = col_condition,out_dir = out_dir,
                    title = "E_pctGFP+")
# Plot % neg
plot_pct_bar_points(summary_df, pct = pct_negative, palette = col_condition,out_dir = out_dir, 
                    title = "E_pctnegative", y_max = 40)
# Plot % double+
plot_pct_bar_points(summary_df, pct = pct_double_GATA3_NANOG, palette = col_condition,out_dir = out_dir, 
                    title = "E_pct_GATA3+_NANOG+", y_max = 40)



In [ ]:
## reformat for comparison with human embryos
plot_pct_bar_points <- function(
  data,                               
  pct = pct_GATA3,                    
  condition = condition,              
  out_dir,                    
  title = NULL,                       
  palette = "black",                     
  w = 1, h = 2,
  y_max = 110,
  y_ticks = 5,
  bar_width = 0.6,
  error_bar_width = 0.2,
  point_size = 0.5,
  point_alpha = 0.7,
  jitter_width = 0.1
) {
  pct       <- enquo(pct)
  condition <- enquo(condition)

  if (is.null(title)) {
    title <- paste0(as_label(pct), "+")
  }

  # 1. Calculate Summary Stats by CONDITION (Mean & SD)
  cond_summary <- data %>%
    group_by(!!condition) %>%
    summarise(
      mean_val = mean(!!pct, na.rm = TRUE),
      sd_val   = sd(!!pct, na.rm = TRUE),
      .groups  = "drop"
    )

  # 2. Build Plot
  # We map X to the Condition. 
  # Note: ensure your 'condition' column levels are set correctly before running this if you want specific order.
  p <- ggplot(data, aes(x = !!condition, y = !!pct)) +
    
    # A. The Bar (Mean of the condition)
    geom_col(
      data = cond_summary,
      aes(y = mean_val, fill = !!condition),
      width = bar_width,
      alpha = 0.6,           # Slight transparency to see points better
      show.legend = FALSE,
      fill = "grey"
    ) +
    
    # B. The Error Bars (Mean +/- SD)
    geom_errorbar(
      data = cond_summary,
      aes(
        y = mean_val, 
        ymin = pmax(0, mean_val - sd_val), # pmax(0, ...) prevents error bar going below 0
        ymax = mean_val + sd_val
      ),
      width = error_bar_width
    ) +
    
    # C. The Individual Points (Jittered)
    geom_jitter(
      size = point_size, 
      alpha = point_alpha,
      width = jitter_width,
      height = 0,             # Don't jitter vertically (keeps Y value accurate)
      na.rm = TRUE,
      color =  palette
    ) +
    
    labs(x = "", y = "%", title = title, fill = rlang::as_name(condition)) +
    settheme +
    scale_y_continuous(limits = c(0, y_max), expand = c(0, 0),
                       breaks = scales::pretty_breaks(y_ticks)) +
      #scale_fill_manual(values=col_condition)+
    theme(
      axis.text.x = element_text(angle = 45, hjust = 1),
      legend.position = "none"
    ) 

  safe_title <- gsub("[^[:alnum:]_\\-]+","_", title)
  ggsave(file.path(out_dir, sprintf("B_%s_by_condition.pdf", safe_title)),
                  plot = p, width = w, height = h)
  options(repr.plot.width=w, repr.plot.height=h)
  p
}

In [ ]:
col_condition_3 = c("D6_RrevG_D" = "#285F62", 
               "D6_RrevG_F" = "#CA4F33")

In [ ]:
head(summary_df %>% filter(sample_name %in% c("D6_RrevG_D", "D6_RrevG_F")))

In [ ]:
# Plot % double+
plot_pct_bar_points(summary_df %>% filter(sample_name %in% c("D6_RrevG_D", "D6_RrevG_F")), condition = sample_name, 
                    pct = pct_negative, palette = col_unspecified, out_dir = out_dir, 
                    title = "pct_negative", y_max = 40)

In [ ]:
check_test <- function(
  data,
  group_var = "sample_name",
  value_var = "pct_negative",
  conditions = c("D6_RrevG_D", "D6_RrevG_F"),
  alpha = 0.05
) {

  g <- data %>%
    filter(.data[[group_var]] %in% conditions) %>%
    split(.[[group_var]]) %>%
    lapply(function(d) na.omit(d[[value_var]]))

  g <- g[conditions]

  n1 <- length(g[[1]])
  n2 <- length(g[[2]])

  if (n1 < 3 || n2 < 3) {
    return(tibble(
      group1 = conditions[1],
      group2 = conditions[2],
      n1 = n1,
      n2 = n2,
      shapiro_p1 = NA_real_,
      shapiro_p2 = NA_real_,
      variance_p = NA_real_,
      normal = NA,
      recommended = "Too few points to assess normality"
    ))
  }

  sp1 <- tryCatch(shapiro.test(g[[1]])$p.value,
                  error = function(e) NA_real_)
  sp2 <- tryCatch(shapiro.test(g[[2]])$p.value,
                  error = function(e) NA_real_)

  normal <- all(!is.na(c(sp1, sp2))) &&
    sp1 > alpha &&
    sp2 > alpha

  variance_p <- tryCatch(
    var.test(g[[1]], g[[2]])$p.value,
    error = function(e) NA_real_
  )

  tibble(
    group1 = conditions[1],
    group2 = conditions[2],
    n1 = n1,
    n2 = n2,
    shapiro_p1 = sp1,
    shapiro_p2 = sp2,
    variance_p = variance_p,
    normal = normal,
    recommended = ifelse(
      normal,
      "Welch t-test",
      "Wilcoxon rank-sum test"
    )
  )
}

check_test(summary_df)

In [ ]:
wilcox_res <- summary_df %>%
  filter(sample_name %in% c("D6_RrevG_D", "D6_RrevG_F")) %>%
  mutate(sample_name = droplevels(sample_name)) %>%
  rstatix::wilcox_test(pct_negative ~ sample_name) %>%
  mutate(
    p_use = p,
    stars = case_when(
      p_use < 0.0001 ~ "****",
      p_use < 0.001  ~ "***",
      p_use < 0.01   ~ "**",
      p_use < 0.05   ~ "*",
      TRUE           ~ "ns"
    )
  )

wilcox_res

In [ ]:
# Plot % double+
plot_pct_bar_points(summary_df %>% filter(sample_name %in% c("D6_RrevG_D", "D6_RrevG_F")), condition = sample_name, 
                    pct = pct_GATA3, palette = col_GATA3, out_dir = out_dir, 
                    title = "pct_GATA3", y_max = 100)

In [ ]:
wilcox_res <- summary_df %>%
  filter(sample_name %in% c("D6_RrevG_D", "D6_RrevG_F")) %>%
  mutate(sample_name = droplevels(sample_name)) %>%
  rstatix::wilcox_test(pct_GATA3 ~ sample_name) %>%
  mutate(
    p_use = p,
    stars = case_when(
      p_use < 0.0001 ~ "****",
      p_use < 0.001  ~ "***",
      p_use < 0.01   ~ "**",
      p_use < 0.05   ~ "*",
      TRUE           ~ "ns"
    )
  )

wilcox_res

In [ ]:
# Plot % double+
plot_pct_bar_points(summary_df %>% filter(sample_name %in% c("D6_RrevG_D", "D6_RrevG_F")), condition = sample_name, 
                    pct = pct_NANOG, palette = col_NANOG, out_dir = out_dir, 
                    title = "pct_NANOG", y_max = 100)

In [ ]:
wilcox_res <- summary_df %>%
  filter(sample_name %in% c("D6_RrevG_D", "D6_RrevG_F")) %>%
  mutate(sample_name = droplevels(sample_name)) %>%
  rstatix::wilcox_test(pct_NANOG ~ sample_name) %>%
  mutate(
    p_use = p,
    stars = case_when(
      p_use < 0.0001 ~ "****",
      p_use < 0.001  ~ "***",
      p_use < 0.01   ~ "**",
      p_use < 0.05   ~ "*",
      TRUE           ~ "ns"
    )
  )

wilcox_res

### F) Intensity per subset

In [ ]:
head(merged_df)

In [ ]:
gata3_pos <- merged_df %>% filter(GATA3pos == TRUE)

In [ ]:
title   <- "F_GATA3_norm_intensity_in GATA3pos"
w <- 3; h <- 4
options(repr.plot.width=w, repr.plot.height=h)
palette <- col_condition

p <- ggplot(gata3_pos, aes(x = forcats::fct_rev(sample_name), y = GATA3_norm_ctr, color = condition)) +
  # boxplots per condition (behind points)
  geom_boxplot(
    aes(fill = condition),
    width = 0.5, alpha = 0.25, outlier.shape = NA,
    position = position_dodge2(width = 0.5, preserve = "single")
  ) +
  # points with jitter+dodge (no width/height here)
  geom_point(
    size = 0.3, alpha = 0.6,
    position = position_jitterdodge(jitter.width = 0.2, dodge.width = 0.5),
    na.rm = TRUE
  ) +
  labs(x = "", y = "GATA3 intensity (normalized)", title = title) +
  settheme +
  #scale_y_continuous(limits = c(0, NA), expand = c(0, 0)) +
  coord_flip() +
  scale_color_manual(values = palette) +
  scale_fill_manual(values = palette, guide = "none")  +
  theme(legend.position = "none")

ggsave(file.path(out_dir, sprintf("%s.pdf", title)), plot = p, width = w, height = h)
p


In [ ]:
nanog_pos <- merged_df %>% filter(NANOGpos == TRUE)

In [ ]:
title   <- "F_NANOG_norm_intensity_in_NANOG_pos"
w <- 3; h <- 4
options(repr.plot.width=w, repr.plot.height=h)
palette <- col_condition

p <- ggplot(nanog_pos, aes(x = forcats::fct_rev(sample_name), y = NANOG_norm_ctr, color = condition)) +
  # boxplots per condition (behind points)
  geom_boxplot(
    aes(fill = condition),
    width = 0.5, alpha = 0.25, outlier.shape = NA,
    position = position_dodge2(width = 0.5, preserve = "single")
  ) +
  # points with jitter+dodge (no width/height here)
  geom_point(
    size = 0.5, alpha = 0.6,
    position = position_jitterdodge(jitter.width = 0.2, dodge.width = 0.5),
    na.rm = TRUE
  ) +
  labs(x = "", y = "NANOG intensity (normalized)", title = title) +
  settheme +
  #scale_y_continuous(limits = c(0, NA), expand = c(0, 0)) +
  coord_flip() +
  scale_color_manual(values = palette) +
  scale_fill_manual(values = palette, guide = "none")  +
  theme(legend.position = "none")


ggsave(file.path(out_dir, sprintf("%s.pdf", title)), plot = p, width = w, height = h)
p


## Intensities of markers

In [ ]:
# thresholds (edit if you want different cutoffs)
gata3_pos_sub <- gata3_pos %>% filter(condition_2 == 'mosaic') %>%filter(timepoint == 'D6') %>%
  group_by(image, sample_name, condition_2, timepoint, state) %>%
  summarise(
    n = n(),
    GATA3 = mean(GATA3_norm_ctr, na.rm = TRUE),
    NANOG = mean(NANOG_norm_ctr, na.rm = TRUE),
    #GATA4 = mean(GATA4_norm_ctr, na.rm = TRUE),
    .groups = "drop"
  )

head(gata3_pos_sub)

In [ ]:
title <- "GATA3"
w <- 1
h <- 2
options(repr.plot.width = w, repr.plot.height = h)

p <- ggplot(
  gata3_pos_sub,
  aes(x = state, y = GATA3, group = state)
) +
  geom_boxplot(
    width = 0.5,
    color = "grey50",
    fill = "grey85",
    alpha = 0.4,
    outlier.shape = NA
  ) +
  geom_point(
    color = col_GATA3,
    size = 0.2,
    alpha = 0.6,
    position = position_jitter(width = 0.15, height = 0),
    na.rm = TRUE
  ) +
  labs(
    x = "",
    y = "GATA3 intensity (normalized)",
    title = title
  ) +
  settheme +
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1),
    legend.position = "none"
  )

ggsave(
  file.path(out_dir, sprintf("D_%s.pdf", title)),
  plot = p,
  width = w,
  height = h
)

p

In [ ]:
wilcox_res <- gata3_pos_sub %>%
  filter(sample_name %in% c("D6_RrevG_D", "D6_RrevG_F")) %>%
  mutate(sample_name = droplevels(sample_name)) %>%
  rstatix::wilcox_test(GATA3 ~ sample_name) %>%
  mutate(
    p_use = p,
    stars = case_when(
      p_use < 0.0001 ~ "****",
      p_use < 0.001  ~ "***",
      p_use < 0.01   ~ "**",
      p_use < 0.05   ~ "*",
      TRUE           ~ "ns"
    )
  )

wilcox_res

In [ ]:
# thresholds (edit if you want different cutoffs)
nanog_pos_sub <- nanog_pos %>% filter(condition_2 == 'mosaic') %>%filter(timepoint == 'D6') %>%
  group_by(image, sample_name, condition_2, timepoint, state) %>%
  summarise(
    n = n(),
    GATA3 = mean(GATA3_norm_ctr, na.rm = TRUE),
    NANOG = mean(NANOG_norm_ctr, na.rm = TRUE),
    #GATA4 = mean(GATA4_norm_ctr, na.rm = TRUE),
    .groups = "drop"
  )

head(nanog_pos_sub)

In [ ]:
title <- "NANOG"
w <- 1
h <- 2
options(repr.plot.width = w, repr.plot.height = h)

p <- ggplot(
  nanog_pos_sub ,
  aes(x = state, y = NANOG, group = state)
) +
  geom_boxplot(
    width = 0.5,
    color = "grey50",
    fill = "grey85",
    alpha = 0.4,
    outlier.shape = NA
  ) +
  geom_point(
    color = col_NANOG,
    size = 0.2,
    alpha = 0.6,
    position = position_jitter(width = 0.15, height = 0),
    na.rm = TRUE
  ) +
  labs(
    x = "",
    y = "NANOG intensity (normalized)",
    title = title
  ) +
  settheme +
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1),
    legend.position = "none"
  )

ggsave(
  file.path(out_dir, sprintf("D_%s.pdf", title)),
  plot = p,
  width = w,
  height = h
)

p

In [ ]:
wilcox_res <- nanog_pos_sub  %>%
  filter(sample_name %in% c("D6_RrevG_D", "D6_RrevG_F")) %>%
  mutate(sample_name = droplevels(sample_name)) %>%
  rstatix::wilcox_test(NANOG ~ sample_name) %>%
  mutate(
    p_use = p,
    stars = case_when(
      p_use < 0.0001 ~ "****",
      p_use < 0.001  ~ "***",
      p_use < 0.01   ~ "**",
      p_use < 0.05   ~ "*",
      TRUE           ~ "ns"
    )
  )

wilcox_res

### G) GFP intensity per celltype (positive marker)

In [ ]:
#% of GFP+ cells in each lineage

summary_df <- merged_df %>%
  group_by(exp, timepoint,image, sample_name, condition, state) %>%
  summarise(
    c_GATA3     = sum(GATA3pos, na.rm = TRUE),
    c_NANOG     = sum(NANOGpos, na.rm = TRUE),
    c_neg     = sum(!GATA3pos & !NANOGpos, na.rm = TRUE),
    c_GATA3_GFP     = sum(GATA3pos & GFPpos, na.rm = TRUE),
    c_NANOG_GFP  = sum(NANOGpos & GFPpos, na.rm = TRUE),
    c_neg_GFP     = sum(!GATA3pos & !NANOGpos & GFPpos, na.rm = TRUE),
    .groups = "drop"
  )

summary_df$GATA3 = summary_df$c_GATA3_GFP / summary_df$c_GATA3
summary_df$NANOG = summary_df$c_NANOG_GFP / summary_df$c_NANOG
summary_df$double_neg = summary_df$c_neg_GFP / summary_df$c_neg


In [ ]:
head(summary_df)

In [ ]:
merged_long <- summary_df  %>%
  pivot_longer(
    cols = c(GATA3, NANOG, double_neg),
    names_to = "marker",
    values_to = "perGFPpos"
  )

In [ ]:
title = "G_perGFPpos_marker"
w <- 7
h <- 2
options(repr.plot.width=w, repr.plot.height=h)
  
p = ggplot(merged_long, aes(x = marker, y = perGFPpos, group = state)) +  # dots for each file
    stat_summary(aes(fill = state),
      fun = mean, 
      geom = "bar",
      position = position_dodge(width = 0.75),
      alpha = 0.4, width = 0.6) +   # error bars
    geom_jitter(
      aes(color = condition),
      position = position_jitterdodge(jitter.width = 0.1, dodge.width = 0.75),
      size = 0.3, alpha = 0.8
    )  +
    labs(
      title = title,
      y = "%GFP",
      x = ""
    )+ settheme +
    theme(
      axis.text.x = element_text(angle = 45, hjust = 1),
      #legend.position = "none"
    )+
      scale_y_continuous(limits = c(0,1.1), expand = c(0, 0))+ 
      facet_wrap(timepoint~condition, nrow = 1)+
      scale_color_manual(values=col_condition)+
  scale_fill_manual(
    values = c(Developed = "grey30", Failed = "grey60"),
    name = "state"
  ) 

ggsave(file.path(out_dir, sprintf("%s.pdf", title)),
                plot = p, width = w, height = h)

p

In [ ]:
title = "G_perGFPpos_marker_sub"
w <- 2
h <- 1.8
options(repr.plot.width=w, repr.plot.height=h)

merged_long_sub = merged_long %>% subset(condition == "Grev_R")

p = ggplot(merged_long_sub, aes(x = marker, y = perGFPpos, group = state)) +  # dots for each file
    stat_summary(aes(fill = state),
      fun = mean, 
      geom = "bar",
      position = position_dodge(width = 0.75),
      alpha = 0.4, width = 0.6) +   # error bars
    geom_jitter(
      aes(color = condition),
      position = position_jitterdodge(jitter.width = 0.1, dodge.width = 0.75),
      size = 0.3, alpha = 0.8
    )  +
    labs(
      title = title,
      y = "%GFP",
      x = ""
    )+ settheme+
    theme(
      axis.text.x = element_text(angle = 45, hjust = 1),
      legend.position = "none"
    ) +
      scale_y_continuous(limits = c(0,1.1), expand = c(0, 0))+ 
      facet_wrap(timepoint~condition, nrow = 1)+
      scale_color_manual(values=col_condition)+
  scale_fill_manual(
    values = c(Developed = "grey30", Failed = "grey60"),
    name = "state"
  ) 

ggsave(file.path(out_dir, sprintf("%s.pdf", title)),
                plot = p, width = w, height = h)

p

### H) Intensity GFP+/-

In [ ]:
title   <- "H_GATA3norm_intensity_in_GATA3pos_GFP"
w <- 4; h <- 2
options(repr.plot.width=w, repr.plot.height=h)
palette <- col_condition

p <- ggplot(gata3_pos, aes(x = forcats::fct_rev(condition), y = GATA3_norm_ctr)) +
  # points with jitter+dodge (no width/height here)
  geom_jitter(aes(color = GFPpos),
    size = 0.3, alpha = 0.6,
    position = position_jitterdodge(jitter.width = 0.2, dodge.width = 0.6),
    na.rm = TRUE
  ) +
  geom_boxplot(
    aes(fill = GFPpos),
    width = 0.5, alpha = 0.25, outlier.shape = NA,
    position = position_dodge(width = 0.6)
  ) +
  labs(x = "", y = "GATA3 intensity (normalized)", title = title) +
  facet_wrap(~state)+
  settheme +
  #scale_y_continuous(limits = c(0, NA), expand = c(0, 0)) +
  #coord_flip() 
  scale_color_manual(values = col_GFP) +
  scale_fill_manual(values = palette, guide = "none") +
  theme( axis.text.x = element_text(angle = 45, hjust = 1), legend.position = "none")

ggsave(file.path(out_dir, sprintf("%s.pdf", title)), plot = p, width = w, height = h)
p

In [ ]:
title   <- "H_NANOG_norm_intensity_in_NANOGpos_GFP"
w <- 4; h <- 2
options(repr.plot.width=w, repr.plot.height=h)
palette <- col_condition

p <- ggplot(nanog_pos, aes(x = forcats::fct_rev(condition), y = NANOG_norm_ctr)) +
  # points with jitter+dodge (no width/height here)
  geom_jitter(aes(color = GFPpos),
    size = 0.3, alpha = 0.6,
    position = position_jitterdodge(jitter.width = 0.2, dodge.width = 0.6),
    na.rm = TRUE
  ) +
  geom_boxplot(
    aes(fill = GFPpos),
    width = 0.5, alpha = 0.25, outlier.shape = NA,
    position = position_dodge(width = 0.6)
  ) +
  labs(x = "", y = "NANOG intensity (normalized)", title = title) +
  settheme +
  facet_wrap(~state)+
  #scale_y_continuous(limits = c(0, NA), expand = c(0, 0)) +
  #coord_flip() 
  scale_color_manual(values = col_GFP) +
  scale_fill_manual(values = palette, guide = "none")  +
  theme(legend.position = "none", axis.text.x = element_text(angle = 45, hjust = 1))

ggsave(file.path(out_dir, sprintf("%s.pdf", title)), plot = p, width = w, height = h)
p

### I) marker positive per gfp

In [ ]:
summary_df <- merged_df %>%
  # 1. Group by all the specified columns
  group_by(exp, timepoint, sample_name, condition, state, image, GFPpos) %>%

  summarise(
    GATA3  = mean(GATA3pos == TRUE, na.rm = TRUE),
    NANOG  = mean(NANOGpos == TRUE, na.rm = TRUE),
    double_neg = mean(GATA3pos == FALSE & NANOGpos == FALSE, na.rm = TRUE),
    n_cells = n(),
    .groups = "drop" # Ungroup to return a clean dataframe
  )

head(summary_df)

In [ ]:
merged_long <- summary_df  %>%
  pivot_longer(
    cols = c(GATA3, NANOG, double_neg),
    names_to = "marker",
    values_to = "prop"
  )%>%
  filter(n_cells > 10)

In [ ]:
title = "I_permarker_pos"
w <- 7
h <- 2
options(repr.plot.width=w, repr.plot.height=h)
  
p = ggplot(merged_long, aes(x = condition, y = prop, group = GFPpos)) +  # dots for each file
    stat_summary(aes(fill = GFPpos),
      fun = mean, 
      geom = "bar",
      position = position_dodge(width = 0.75),
      alpha = 0.4, width = 0.6) +   # error bars
    geom_jitter(
      aes(color = GFPpos),
      position = position_jitterdodge(jitter.width = 0.1, dodge.width = 0.75),
      size = 0.3, alpha = 0.8
    )  +
    labs(
      title = title,
      y = "proportion",
      x = ""
    )+ settheme +
    theme(
      axis.text.x = element_text(angle = 45, hjust = 1),
      #legend.position = "none"
    )+
      scale_y_continuous(limits = c(0,1.1), expand = c(0, 0))+ 
      facet_wrap(timepoint ~ marker, nrow = 1)+
      #scale_color_manual(values=marker)+
      scale_color_manual(values = col_GFP)+
      scale_fill_manual(values = col_GFP)

ggsave(file.path(out_dir, sprintf("%s.pdf", title)),
                plot = p, width = w, height = h)

p

In [ ]:
title = "G_perGFPpos_marker_sub"
w <- 2
h <- 1.8
options(repr.plot.width=w, repr.plot.height=h)

merged_long_sub = merged_long %>% subset(condition == "Grev_R")

p = ggplot(merged_long_sub, aes(x = marker, y = perGFPpos, group = state)) +  # dots for each file
    stat_summary(aes(fill = state),
      fun = mean, 
      geom = "bar",
      position = position_dodge(width = 0.75),
      alpha = 0.4, width = 0.6) +   # error bars
    geom_jitter(
      aes(color = condition),
      position = position_jitterdodge(jitter.width = 0.1, dodge.width = 0.75),
      size = 0.3, alpha = 0.8
    )  +
    labs(
      title = title,
      y = "%GFP",
      x = ""
    )+ settheme+
    theme(
      axis.text.x = element_text(angle = 45, hjust = 1),
      legend.position = "none"
    ) +
      scale_y_continuous(limits = c(0,1.1), expand = c(0, 0))+ 
      facet_wrap(timepoint~condition, nrow = 1)+
      scale_color_manual(values=col_condition)+
  scale_fill_manual(
    values = c(Developed = "grey30", Failed = "grey60"),
    name = "state"
  ) 

ggsave(file.path(out_dir, sprintf("%s.pdf", title)),
                plot = p, width = w, height = h)

p

## Save

In [ ]:
head(merged_df)

In [ ]:
write.csv(merged_df, file.path(out_dir, sprintf("%s_analysis_summary.csv", "merged")), row.names = FALSE)